# Global Tech Job Market EDA (2019-2026)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('tech_job_market_2019_2026.csv')
df.shape

## 1. Structural checks

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 2. Missing values

In [ ]:
df.isnull().sum()

In [ ]:
# check for placeholder words that mean missing but are not NaN
placeholder_words = ['unknown', 'tbd', 'not disclosed', 'n/a', 'na', 'none', '']

for col in df.select_dtypes(include='object').columns:
    count = 0
    for word in placeholder_words:
        count = count + (df[col].astype(str).str.strip().str.lower() == word).sum()
    if count > 0:
        print(col, ':', count)

In [ ]:
# check if missing city is linked to remote jobs
df[df['job_location_city'].isnull()]['work_mode'].value_counts()

In [ ]:
# check if missing filled date is linked to jobs that are not filled yet
df[df['date_position_filled'].isnull()]['position_status'].value_counts()

## 3. Duplicates

In [ ]:
print('duplicate rows:', df.duplicated().sum())
print('duplicate job_id:', df['job_id'].duplicated().sum())

In [ ]:
# extra spaces in company names
df['company_name'].str.strip().ne(df['company_name']).sum()

## 4. Fixing categories

In [ ]:
df2 = df.copy()

In [ ]:
df['work_mode'].value_counts()

In [ ]:
# map every spelling to one clean value using a dictionary + a loop
work_mode_map = {}
for v in df['work_mode'].dropna().unique():
    key = v.strip().lower()
    if key in ['remote', 'work from home', 'wfh']:
        work_mode_map[v] = 'Remote'
    elif key == 'hybrid':
        work_mode_map[v] = 'Hybrid'
    elif key in ['onsite', 'on-site', 'in-office']:
        work_mode_map[v] = 'Onsite'
    else:
        work_mode_map[v] = np.nan

df2['work_mode'] = df2['work_mode'].map(work_mode_map)
df2['work_mode'].value_counts()

In [ ]:
df['visa_sponsorship_offered'].value_counts()

In [ ]:
yes_no_map = {
    'Y': True, 'Yes': True, 'True': True, '1': True, 1: True, True: True,
    'N': False, 'No': False, 'False': False, '0': False, 0: False, False: False,
}
df2['visa_sponsorship_offered'] = df2['visa_sponsorship_offered'].map(yes_no_map)
df2['layoff_within_12_months_flag'] = df2['layoff_within_12_months_flag'].map(yes_no_map)
df2['visa_sponsorship_offered'].value_counts()

In [ ]:
df['sector'].value_counts()

In [ ]:
sector_map = {
    'HealthTech': 'HealthTech', 'healthtech': 'HealthTech', 'Healthtech': 'HealthTech',
    'AI/ML': 'AI/ML', 'ai/ml': 'AI/ML', 'Ai/Ml': 'AI/ML',
    'SaaS': 'SaaS', 'saas': 'SaaS', 'Saas': 'SaaS',
    'Fintech': 'Fintech', 'fintech': 'Fintech', 'FinTech': 'Fintech',
    'E-commerce': 'E-commerce', 'e-commerce': 'E-commerce', 'E-Commerce': 'E-commerce',
    'Cloud/Infra': 'Cloud/Infra', 'cloud/infra': 'Cloud/Infra',
    'Cybersecurity': 'Cybersecurity', 'cybersecurity': 'Cybersecurity',
    'Gaming': 'Gaming', 'gaming': 'Gaming',
    'EdTech': 'EdTech', 'edtech': 'EdTech', 'Edtech': 'EdTech',
    'Crypto/Web3': 'Crypto/Web3', 'crypto/web3': 'Crypto/Web3',
    'Social/AdTech': 'Social/AdTech', 'social/adtech': 'Social/AdTech', 'Social/Adtech': 'Social/AdTech',
    'Logistics': 'Logistics', 'logistics': 'Logistics',
}
df2['sector'] = df2['sector'].str.strip().map(sector_map)
df2['sector'].value_counts()

In [ ]:
df2['company_name'] = df2['company_name'].str.strip()

In [ ]:
df['salary_min_usd'].head(10)

In [ ]:
# clean salary text -> number, step by step
salary_min = df2['salary_min_usd'].astype(str)
salary_min = salary_min.str.replace('$', '', regex=False)
salary_min = salary_min.str.replace('USD', '', regex=False)
salary_min = salary_min.str.replace(',', '', regex=False)
salary_min = salary_min.str.strip()
salary_min = salary_min.replace(['Not Disclosed', 'nan', 'N/A', 'Unknown'], np.nan)
df2['salary_min_clean'] = pd.to_numeric(salary_min, errors='coerce')
df2.loc[df2['salary_min_clean'] < 0, 'salary_min_clean'] = np.nan

salary_max = df2['salary_max_usd'].astype(str)
salary_max = salary_max.str.replace('$', '', regex=False)
salary_max = salary_max.str.replace('USD', '', regex=False)
salary_max = salary_max.str.replace(',', '', regex=False)
salary_max = salary_max.str.strip()
salary_max = salary_max.replace(['Not Disclosed', 'nan', 'N/A', 'Unknown'], np.nan)
df2['salary_max_clean'] = pd.to_numeric(salary_max, errors='coerce')
df2.loc[df2['salary_max_clean'] < 0, 'salary_max_clean'] = np.nan

df2[['salary_min_clean', 'salary_max_clean']].describe()

In [ ]:
df2['primary_programming_language'] = df2['primary_programming_language'].replace(
    {'Unknown': np.nan, 'TBD': np.nan}
)

In [ ]:
print('rows before:', len(df2))
df2 = df2.drop_duplicates()
print('rows after:', len(df2))

## 5. Logical checks

In [ ]:
issue1 = df2['years_experience_min'] > df2['years_experience_max']
issue2 = df2['offers_accepted'] > df2['offers_extended']
issue3 = df2['offers_extended'] > df2['number_of_applicants']
issue4 = df2['number_of_applicants'] < 0

print('min exp > max exp:', issue1.sum())
print('accepted > extended:', issue2.sum())
print('extended > applicants:', issue3.sum())
print('negative applicants:', issue4.sum())

In [ ]:
df2['posting_date'] = pd.to_datetime(df2['posting_date'], errors='coerce')
df2['application_deadline'] = pd.to_datetime(df2['application_deadline'], errors='coerce')
df2['date_position_filled'] = pd.to_datetime(df2['date_position_filled'], errors='coerce')

issue5 = df2['posting_date'] > df2['date_position_filled']
print('posted after filled:', issue5.sum())

In [ ]:
df2['has_logic_issue'] = issue1 | issue2 | issue3 | issue4 | issue5.fillna(False)
df2['has_logic_issue'].sum()

## 6. Distributions and outliers

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df2['company_size_employees'], bins=40)
plt.title('Company size')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df2['salary_min_clean'].dropna(), bins=40)
plt.title('Salary min (USD)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df2['number_of_applicants'])
plt.title('Applicants (checking outliers)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df2['number_of_interview_rounds'])
plt.title('Interview rounds (checking outliers)')
plt.show()

## 7. Comparing variables

In [ ]:
plt.figure(figsize=(8, 4))
df2['sector'].value_counts().plot(kind='bar')
plt.title('Postings by sector')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
df2['company_type'].value_counts().plot(kind='bar')
plt.title('Postings by company type')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
df2['job_role'].value_counts().head(10).plot(kind='barh')
plt.title('Top 10 job roles')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
df2['position_status'].value_counts().plot(kind='bar')
plt.title('Position status (hiring outcome)')
plt.show()

In [ ]:
order = ['Intern', 'Entry', 'Mid', 'Senior', 'Lead', 'Principal']
plt.figure(figsize=(8, 5))
sns.boxplot(data=df2, x='seniority_level', y='salary_min_clean', order=order)
plt.title('Salary by seniority')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df2, x='company_type', y='salary_min_clean')
plt.title('Salary by company type')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df2, x='work_mode', y='salary_min_clean')
plt.title('Salary by work mode')
plt.show()

In [ ]:
# interview rounds vs hiring outcome
plt.figure(figsize=(8, 5))
sns.boxplot(data=df2, x='position_status', y='number_of_interview_rounds')
plt.title('Interview rounds by position status')
plt.show()

In [ ]:
numeric_columns = ['years_experience_min', 'years_experience_max', 'number_of_applicants',
                    'number_of_interview_rounds', 'offers_extended', 'offers_accepted',
                    'salary_min_clean', 'salary_max_clean', 'company_size_employees']

plt.figure(figsize=(8, 6))
sns.heatmap(df2[numeric_columns].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation heatmap')
plt.show()

## 8. Trend over time

In [ ]:
postings_per_month = df2.set_index('posting_date').resample('ME').size()

plt.figure(figsize=(12, 4))
postings_per_month.plot()
plt.title('Postings per month')
plt.show()

## 9. Feature engineering

In [ ]:
# time to fill = days between posting and filling
df2['time_to_fill_days'] = (df2['date_position_filled'] - df2['posting_date']).dt.days
df2.loc[df2['time_to_fill_days'] < 0, 'time_to_fill_days'] = np.nan
df2['time_to_fill_days'].describe()

In [ ]:
# offer accept ratio = accepted / extended
df2['offer_accept_ratio'] = df2['offers_accepted'] / df2['offers_extended']
df2.loc[df2['offers_extended'] == 0, 'offer_accept_ratio'] = np.nan
df2['offer_accept_ratio'].describe()

In [ ]:
# average salary = midpoint of min and max
df2['avg_salary_usd'] = (df2['salary_min_clean'] + df2['salary_max_clean']) / 2
df2['avg_salary_usd'].describe()

In [ ]:
# tech stack as a list + count, and most common technologies overall
df2['tech_stack_list'] = df2['required_tech_stack'].str.split(', ')
df2['tech_stack_count'] = df2['tech_stack_list'].apply(len)

top_tech = df2['tech_stack_list'].explode().value_counts().head(10)

plt.figure(figsize=(8, 5))
top_tech.plot(kind='barh')
plt.title('Top 10 technologies')
plt.gca().invert_yaxis()
plt.show()

## 10. Summary of findings

| Issue | Count | What was done |
|---|---|---|
| Dates stored as text | 3 columns | Converted to datetime |
| Salary stored as text ($, commas, USD) | ~111,682 values | Cleaned to numbers |
| -1 placeholder in salary | 737 | Treated as missing |
| "Not Disclosed" in salary | 686 | Treated as missing |
| "Unknown"/"TBD" in programming language | 1,093 | Treated as missing |
| Missing city | 25,107 rows | Left missing, mostly remote jobs |
| Missing filled date | 25,781 rows | Left missing, job not filled yet |
| work_mode spelled many ways | ~5,600 rows | Mapped to 3 clean values |
| visa/layoff mixed yes-no spelling | all rows | Mapped to True/False |
| sector case differences | 24 -> 12 categories | Mapped to clean values |
| extra spaces in company name | 1,694 rows | Stripped |
| duplicate rows | 1,193 rows | Dropped |
| same job_id, different postings | 807 ids | Kept, noted as a data issue |
| min exp > max exp | 315 rows | Flagged, not deleted |
| accepted offers > extended offers | 597 rows | Flagged, not deleted |
| extended offers > applicants | 374 rows | Flagged, not deleted |
| negative applicants | 247 rows | Flagged, not deleted |
| posted after filled date | 445 rows | Flagged, excluded from time_to_fill_days |

Notes: no missing values were filled in anywhere, they were left as missing. Rows with logic
issues were kept but their broken values were not used in calculations.